In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
zomato_data = pd.read_csv("/kaggle/input/datasets/himanshupoddar/zomato-bangalore-restaurants/zomato.csv")
print(zomato_data.shape)
zomato_data.head()

In [ ]:
# ### Restaurant details what do we have per row (grain)
# zomato_data.head()
# zomato_data.info()
# concat = zomato_data['name'] + "_" + zomato_data['address'] + "_" + zomato_data['listed_in(type)'] +"_" + zomato_data['listed_in(city)'] 
# concat.value_counts()
# zomato_data[zomato_data['name'] == "Cinnamon"]
# a, b = zomato_data.loc[16406], zomato_data.loc[16836]
# a[a != b]

# ### What is the difference between "location" and "listed_in(city)"
# zomato_data['location'],zomato_data['listed_in(city)']
# print(zomato_data['location'].value_counts())
# zomato_data['listed_in(city)'].value_counts()
# zomato_data[(zomato_data['location']=='BTM') & (zomato_data['listed_in(city)']=='BTM')]
# zomato_data[zomato_data['location'] != zomato_data['listed_in(city)']]

# ### Null values within the rate and approx_cost and if we can handle them
# zomato_data['rate'].describe()
# zomato_data['approx_cost(for two people)'].describe()

### To look into the grain and duplicates
# zomato_data.duplicated().sum()
# zomato_data.loc[[16406, 16836]].T
# zomato_data['rate'].info()
zomato_data[(zomato_data["votes"] > 0) & (zomato_data["rate"].isna())]
# bad = ~zomato_data["rate"].str.endswith("/5", na=False)
# zomato_data.loc[bad, "rate"].value_counts(dropna=False)
# zomato_data[(zomato_data['approx_cost(for two people)'].str.replace(",", "").astype(float) > 4000)]
zomato_data.shape

In [ ]:
restaurants=zomato_data.drop_duplicates(subset=["name", "address"]).copy()
restaurants = restaurants.drop(columns=["url", "phone", "reviews_list", "menu_item"])
restaurants.shape
restaurants.rate.info()
# 12499-10094

In [ ]:
### Convert rate to float
restaurants['rate'] = restaurants['rate'].replace("NEW",np.nan)
restaurants['rate'] = restaurants['rate'].replace("-",np.nan)
restaurants['rate'] = restaurants['rate'].str.replace("/5","").astype(float)
# zomato_data['rate'].value_counts()
# restaurants.rate.info

In [ ]:
restaurants.rate.info()

In [ ]:
### Convert cost to float
restaurants['approx_cost(for two people)'] = restaurants['approx_cost(for two people)'].str.replace(",","").astype(float)
restaurants["approx_cost(for two people)"].isna().sum()
# restaurants.shape

In [ ]:
area_summary = restaurants.groupby("location").agg(
    restaurants=("name", "count"),
    median_cost=("approx_cost(for two people)", "median"),
    median_rating=("rate", "median"),
)
area_summary.head()

## Audit

One row is a restaurant × listed_in(type) × listed_in(city), so 51,717 rows are not 51,717 restaurants. Restaurant-level questions use the deduplicated table (name + address): 12,499 restaurants.

| Column | Issue | Decision and why |
|---|---|---|
| name, address | repeated across type and city rows; near-duplicates differ only in reviews_list | fix: keep one row per name + address for restaurant-level questions |
| online_order | Yes/No text | leave: used as the Q2 grouping |
| rate | text like "4.1/5"; "NEW" and "-" (778 restaurants after dedup; at row level 2,208 NEW, always 0 votes, and 69 "-"); plus real NaN | fix: NEW and "-" become NaN, strip "/5", convert to float; leave NaN unfilled so no ratings are invented |
| votes | integer, no issues | leave |
| location | restaurant's own neighbourhood | leave: used as "area" in Q1 |
| listed_in(city) | broader Zomato area list, not a city | leave: not used as area |
| listed_in(type) | one row per service type (Delivery, Dine-out...) | leave: only explains the duplicate rows |
| cuisines | several cuisines in one cell | leave for now: split in Q3 |
| approx_cost(for two people) | text with commas ("1,200"); 59 missing | fix: remove commas, convert to float; leave the 59 NaN; the ~70 restaurants above 3,000 are real and stay |
| url, phone, reviews_list, menu_item | not needed | drop |

## Cleaning log

| Step | Rows before | Rows after | Note |
|---|---|---|---|
| Keep one row per name + address | 51,717 | 12,499 | one row per restaurant |
| Drop unused columns | 17 columns | 13 columns | url, phone, reviews_list, menu_item |
| Convert rate to float | 12,499 | 12,499 | NaN in rate: 2,405 before, 3,183 after (778 were "NEW" or "-"); about 25% of restaurants are unrated, kept for counts and skipped in rating medians |
| Convert cost to float | 12,499 | 12,499 | commas removed; 59 NaN left as is |

In [ ]:
area_summary = restaurants.groupby("location").agg(
    restaurants=("name", "count"),
    median_cost=("approx_cost(for two people)", "median"),
    median_rating=("rate", "median"),
)
area_summary.sort_values("median_cost", ascending=False).head(10)

In [ ]:
res_cut  = area_summary["restaurants"].mean() + area_summary["restaurants"].std()
cost_cut = area_summary["median_cost"].mean() + area_summary["median_cost"].std()
crowded   = area_summary["restaurants"] > res_cut
expensive = area_summary["median_cost"] > cost_cut
area_summary[crowded & expensive]

In [ ]:
crowded   = area_summary["restaurants"] > area_summary["restaurants"].quantile(0.75)
expensive = area_summary["median_cost"] > area_summary["median_cost"].median()
print(area_summary["restaurants"].quantile(0.75))
area_summary[crowded & expensive]       # crowded and expensive


In [ ]:
area_summary[crowded & ~expensive]      # crowded and cheap

In [ ]:
import matplotlib.pyplot as plt
ax = area_summary.plot.scatter(x="restaurants", y="median_cost")
ax.axvline(area_summary["restaurants"].quantile(0.75), linestyle="--")
ax.axhline(area_summary["median_cost"].median(), linestyle="--")
plt.show()

Q1 findings. Of about 93 areas, 23 are “crowded” (top quarter by restaurant count, 151 or more restaurants). Only three of them are also expensive (above the ₹400 median area cost): Indiranagar, Koramangala 5th Block and Kalyan Nagar. They also rate higher (3.8 to 4.0) than the big mid-priced areas such as Whitefield, Electronic City, BTM and HSR (3.5 to 3.7). The very high-cost areas, like Sankey Road and Lavelle Road, are small, so crowding in Bangalore does not drive price. Note: “cheap” here means at or below the typical area median of ₹400, and many areas sit exactly on that line.